In [65]:
import os
import json
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa


# cross
# jsonl_dir = "/home/comp/24483737/he/group1"
# self
# jsonl_dir = "/home/comp/24483737/he/group2"
# model_prefix = "llama2"  

# self
# jsonl_dir = "/home/comp/24483737/he/group3"
# cross
# jsonl_dir = "/home/comp/24483737/he/group4"
# model_prefix = "qwen3"  


# jsonl_dir = "/home/comp/24483737/he"
# model_prefix = "qwen3"  
# model_prefix = "llama2" 

jsonl_dir = "/home/comp/24483737/FastChat/he"
model_prefix = "llama2" 

label_set = ["model_1", "model_2", "tie"]
label2idx = {label: idx for idx, label in enumerate(label_set)}

jsonl_files = sorted([f for f in os.listdir(jsonl_dir) 
                      if f.endswith(".jsonl") and f.startswith(model_prefix) and "pair" in f])
n_judges = len(jsonl_files)

all_data = {}

for f in jsonl_files:
    filepath = os.path.join(jsonl_dir, f)
    with open(filepath, "r", encoding="utf-8") as fin:
        for line in fin:
            obj = json.loads(line)
            qid = obj["question_id"]
            votes = []
            for key in ["g1_winner", "g2_winner"]:
                v = obj.get(key)
                if v in label2idx:
                    votes.append(v)
            if len(votes) != 2:
                continue 
            if qid not in all_data:
                all_data[qid] = []
            all_data[qid].append(votes)

table = []
for qid, judges_votes in all_data.items():
    while len(judges_votes) < n_judges:
        judges_votes.append(['tie', 'tie'])  
    flat_votes = [v for vote_list in judges_votes for v in vote_list]
    row = [flat_votes.count(label) for label in label_set]
    table.append(row)

table = np.array(table)

print("Number of items used:", table.shape[0])
print("Votes per item:", table.sum(axis=1)[0], "(should be 2 * n_judges =", 2*n_judges, ")")

kappa = fleiss_kappa(table)
print("Fleiss Kappa:", kappa)


Number of items used: 855
Votes per item: 6 (should be 2 * n_judges = 6 )
Fleiss Kappa: 0.6317098964491719
